# RQ3b OpenMent — mentor/mentee NK-gap investigation

Thin driver for `scripts/openment_roles.py`. The module holds all logic; this
notebook only calls it and displays results. Phases run **in order** because
each can change the next:

| Phase | Output |
|---|---|
| P0 schema audit (blocking) | `rq3b_om_P0_schema_report.md` |
| P1 hygiene checks | `rq3b_om_P1_unit_flags.csv`, `rq3b_om_P1_rates_by_filter.csv` |
| P2 project overview | `rq3b_om_P2_projects.csv` |
| P3 gap decomposition | `rq3b_om_P3_*.csv`, `rq3b_om_P3_models.md` |
| P4 thread sampling | `rq3b_om_P4_*.csv`, `rq3b_om_threads/*.docx` |
| P5 signal coding | `rq3b_om_P5_*.csv` |

> NOTE: the pattern YAMLs are being actively tuned (2026-10, user decision),
> so the regression anchor may legitimately drift from 51.837 / 39.471.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Markdown, display

from scripts import openment_roles as om

print("module loaded from", om.__file__)

module loaded from /Users/giulianowietig/PycharmProjects/Graph-NK/scripts/openment_roles.py


## P0 — schema audit (blocking)

Confirms the thread key, role/author/date fields and character offsets before
any analysis. Read the report before continuing.

In [2]:
om.run_p0()
display(Markdown((om.OUT_DIR / "rq3b_om_P0_schema_report.md").read_text()))

regression anchor drift (expected while YAMLs are tuned):
- mentor: observed 24.000 vs expected 51.837 → DRIFT
- mentee: observed 16.708 vs expected 39.471 → DRIFT


# RQ3b OpenMent — P0 schema audit

- Rules: version 0.2, 67 active, 9 candidate.
- Artefacts/units loaded: 75,284 threads, 1,335,605 TextUnits.
- Signals under the NK predicate: 737,508.
- Units with a known role: 566,369.

## Regression anchor (unfiltered pooled rates)

| role | units | tokens | nk | rate_per_1k |
| --- | --- | --- | --- | --- |
| mentee | 352631 | 9677476 | 161689 | 16.708 |
| mentor | 213738 | 5729562 | 137507 | 24.0 |

Expected target (pre-tuning rule set): mentor 51.837, mentee 39.471. The pattern YAMLs are being actively fine-tuned (2026-10, user decision), so the observed values may legitimately drift.

- mentor: observed 24.000 vs expected 51.837 → DRIFT
- mentee: observed 16.708 vs expected 39.471 → DRIFT

## Thread key

`thread_id` = `p.record_id` (`issue|pr:openment:<pid>:<n>`). `message_id` = thread + `:op` (title/body) or `:comment:<position>`.
OpenMent is GitHub-shaped: each issue/PR *is* the thread; no reply-graph reconstruction is needed. No Conversation node wraps OpenMent artefacts.

## Dates and provenance

Dates are anonymised day offsets (numeric), not ISO timestamps; the Word export shows relative elapsed days from the thread start.
Author = `u.author_login`; role from `project_roles.csv` (6,422 (project, user) mappings).

## HAS_TEXT roles

| tu_role | units |
| --- | --- |
| comment_body | 1189376 |
| title | 75280 |
| body | 70949 |

## Span-offset audit (random threads)

Offsets are character offsets into `TextUnit.text`. `char`/`char_loose` are exact matches; `search`/`failed` are the ones to watch.

| layer | char | char_unverified | total |
| --- | --- | --- | --- |
| classifier | 0 | 757 | 757 |
| lexical | 145 | 0 | 145 |
| morpho_syntactic | 1642 | 118 | 1760 |
| rhetorical | 14 | 0 | 14 |
| word_formation | 111 | 0 | 111 |


## P1 — hygiene checks

Flags quoted / boilerplate / signature / non-prose units. Nothing is deleted;
rates are reported with and without each filter.

In [3]:
om.run_p1()
display(pd.read_csv(om.OUT_DIR / "rq3b_om_P1_rates_by_filter.csv"))

regression anchor drift (expected while YAMLs are tuned):
- mentor: observed 24.000 vs expected 51.837 → DRIFT
- mentee: observed 16.708 vs expected 39.471 → DRIFT


,filter,role,units,tokens,nk,rate_per_1k
0,none,mentee,352631,9677476,161689,16.707766
1,none,mentor,213738,5729562,137507,23.999566
2,no_quote_markers,mentee,351187,9577782,160882,16.797417
3,no_quote_markers,mentor,212741,5685134,136693,24.043936
4,no_repeats,mentee,279914,8586417,140915,16.411386
5,no_repeats,mentor,177620,4928303,118537,24.052295
6,no_quoted,mentee,278543,8489734,140154,16.508644
7,no_quoted,mentor,176787,4890634,117826,24.092173
8,no_boilerplate,mentee,262943,8416438,139452,16.569005
9,no_boilerplate,mentor,168980,4875382,117491,24.098830


## P2 — project overview (empirical-Bayes shrunk rates)

Per-project, per-role rates with Gamma–Poisson shrinkage and within-project tests.

In [6]:
om.run_p2()
projects = pd.read_csv(om.OUT_DIR / "rq3b_om_P2_projects.csv")
eligible = projects[projects.min_exposure]
print(f"eligible projects (both roles >= 1k tokens): {len(eligible):,}")
display(eligible.sort_values("gap_abs", ascending=False).head(20))

eligible projects (both roles >= 1k tokens): 744


,project_id,units,tokens,nk,n_mentors,n_mentees,n_threads,tokens_mentor,nk_mentor,rate_mentor,tokens_mentee,nk_mentee,rate_mentee,rate_per_1k,rate_eb,rate_eb_mentor,rate_eb_mentee,gap_abs,gap_rel,min_exposure
864,7b6b89e4594c1e911ddba6d870237afff2173a08fd5157...,282,6343,204,1,1,16,2127,123,57.827927,4216,81,19.212524,32.161438,31.218721,51.653299,19.011001,32.642298,2.717022,True
1337,bec38e4b471fa95cd45178f7b8b3941734d2a276c13361...,548,15721,548,2,1,54,4198,245,58.361124,11523,303,26.295236,34.857833,34.375704,54.868848,25.997836,28.871012,2.110516,True
1556,de9633b0a9ffe44e2b67fbb9ed5342818c118f6a41301f...,76,3246,107,2,1,6,1483,77,51.921780,1763,30,17.016449,32.963648,31.134617,45.148821,16.963037,28.185784,2.661600,True
927,84d328fb337b9ed665bb12d08f2efb6d6a4d4b8af2b9bc...,246,5353,154,1,1,24,4340,150,34.562212,1013,4,3.948667,28.768915,27.960328,33.520356,7.354601,26.165755,4.557740,True
1141,a66f592ac97b226dab3555fe14785807b53ecf019a7918...,133,4629,136,2,1,10,3195,123,38.497653,1434,13,9.065551,29.379996,28.397122,36.621457,10.629199,25.992258,3.445364,True
198,1b4bed720a70c1b6b14f112416b7b0cbeb94d26582bad1...,91,2747,59,1,1,14,1422,52,36.568214,1325,7,5.283019,21.477976,21.157356,33.421458,7.771020,25.650438,4.300781,True
790,71cd3853ad68fb21de9659f12cb8b03b7d01dad508c54b...,408,24218,490,1,1,62,4125,177,42.909091,20093,313,15.577564,20.232885,20.216211,40.956756,15.597939,25.358816,2.625780,True
175,195ebd0c29069b38896078c02195589a1d6b40ef59c2fb...,329,7883,79,2,1,35,1430,45,31.468531,6453,34,5.268867,10.021565,10.589240,29.606413,5.887406,23.719007,5.028770,True
905,819cdca1cc593ac41bbd7fb267bda1df0c2543a76d52d9...,352,13006,329,2,1,65,5779,223,38.587991,7227,106,14.667220,25.296017,25.075623,37.480143,14.766316,22.713827,2.538219,True
1263,b378360264c314738e3c875656fb38aa53a466a19f2bf2...,897,20003,469,1,1,76,5278,216,40.924593,14725,253,17.181664,23.446483,23.346978,39.527371,17.170082,22.357289,2.302107,True


## P3 — gap decomposition, skew, confirmatory model

In [7]:
om.run_p3()
contrib = pd.read_csv(om.OUT_DIR / "rq3b_om_P3_rule_contrib.csv")
display(contrib.head(25))
display(Markdown((om.OUT_DIR / "rq3b_om_P3_models.md").read_text()))

/Users/giulianowietig/PycharmProjects/Graph-NK/scripts/openment_roles.py:919: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  family=sm.families.NegativeBinomial(),


,layer,category,subcategory,rule_id,mentee,mentor,rate_mentor,rate_mentee,delta,share_of_gap
0,morpho_syntactic,evidential,inferential,morph.evid.inferential_seem,26188,24926,4.350420,2.706077,1.644342,0.225506
1,morpho_syntactic,hedging,plausibility_shield,morph.hedge.shield_think,30523,26734,4.665976,3.154025,1.511951,0.207350
2,morpho_syntactic,hedging,adverbial,morph.hedge.adverbial_maybe,6765,9894,1.726834,0.699046,1.027788,0.140951
3,morpho_syntactic,hedging,adverbial,morph.hedge.adverbial_perhaps,5890,9172,1.600820,0.608630,0.992191,0.136069
4,morpho_syntactic,modality,epistemic,morph.mod.epistemic_maymight,12106,12618,2.202263,1.250946,0.951317,0.130464
5,rhetorical,comparison,approximator,rhetor.comparison.something_like,4284,5867,1.023988,0.442677,0.581310,0.079721
6,morpho_syntactic,syntactic_pattern,tag_question,morph.syn.tag_question,1007,1478,0.257960,0.104056,0.153904,0.021106
7,morpho_syntactic,syntactic_pattern,embedded_question,morph.syn.embedded_question,432,884,0.154288,0.044640,0.109648,0.015037
8,morpho_syntactic,negation,scoped_epistemic,morph.neg.scoped_epistemic,2479,2071,0.361459,0.256162,0.105297,0.014440
9,morpho_syntactic,syntactic_pattern,embedded_question,morph.syn.i_wonder,263,687,0.119904,0.027177,0.092728,0.012717


# RQ3b P3 — confirmatory models

Message-level aggregates: 519,748 messages, 1,778 projects, 3,278 authors.

## Primary: GEE negative-binomial, exchangeable, clustered by project

- rate ratio (mentor vs mentee) = **1.397** [1.333, 1.463]
- coefficient (log RR) = 0.3340

## Sensitivity GLMM skipped
- 1,778 projects / 3,278 authors exceed the 
size guard; a clustered GEE above is the reported sensitivity.


## P4 — thread sampling frame and Word export

Builds the thread frame, selects the stratified sample and renders the Word
files under `notebooks/rq3b_om_threads/`. Check the alignment report: the share
of `failed` spans must stay below 2%.

In [10]:
om.run_p4()
frame = pd.read_csv(om.OUT_DIR / "rq3b_om_P4_thread_frame.csv")
sample = pd.read_csv(om.OUT_DIR / "rq3b_om_P4_thread_sample.csv")
print(f"eligible threads: {int(frame.eligible.sum()):,} / {len(frame):,}")
display(sample)

eligible threads: 3,935 / 68,182


,thread_id,stratum,selection_reason,seed
0,issue:openment:54d1156417c977d6b769537ccf83ce3...,typical,|gap-median|<0.25SD (median=7.05),20261002
1,issue:openment:d8dc50873b3e5670daab7e62355e103...,typical,|gap-median|<0.25SD (median=7.05),20261002
2,issue:openment:41f9a2a85cd95b3fca21a28c5bffd2a...,typical,|gap-median|<0.25SD (median=7.05),20261002
3,issue:openment:ec43efbed18ebd7846a76716f705d99...,typical,|gap-median|<0.25SD (median=7.05),20261002
4,issue:openment:8f57cea5ade390741d2913a9e91a85d...,typical,|gap-median|<0.25SD (median=7.05),20261002
5,issue:openment:f49ce8d1b336fa5bd8f43eeef0b9782...,extreme_pos,gap_abs=59.32,20261002
6,issue:openment:cf43501004e989ccec0397aa8ce0e83...,extreme_pos,gap_abs=56.74,20261002
7,issue:openment:67505a2aba9161ba84ccf393aba939c...,composition,"similar gap, category cos-dist=1.00",20261002
8,issue:openment:06ae237c9532b70d89271e4bbff6aed...,composition,"similar gap, category cos-dist=1.00",20261002
9,issue:openment:8b167c65f729a1796141389af2af1e7...,same_mentor,lowest-gap thread of a repeated mentor,20261002


## P5 — signal coding sheet

Role × category stratified signal sample with KWIC context. The role column is
kept in a separate key file for blind-ish coding; the function definitions live
in the coding-manual addendum, not in code.

In [7]:
om.run_p5()
sheet = pd.read_csv(om.OUT_DIR / "rq3b_om_P5_coding_sheet.csv")
print(f"signals to code: {len(sheet):,}")
display(sheet.head(10))

signals to code: 572


,signal_id,tu_id,thread_id,kwic_left,span,kwic_right,context,role_hidden,function,epistemic_subject,note
0,issue:openment:42d138a91ef246aeb669b28276304b6...,issue:openment:42d138a91ef246aeb669b28276304b6...,issue:openment:42d138a91ef246aeb669b28276304b6...,results differ\r 3. If I bypass whitening in M...,inverse,would do - `` return the same inverse in this...,First insights from parallel Python/MATLAB deb...,NaN,NaN,NaN,NaN
1,issue:openment:922979aa5698e0042e54e1f344dc787...,issue:openment:922979aa5698e0042e54e1f344dc787...,issue:openment:922979aa5698e0042e54e1f344dc787...,I have added a check so as to solve the inequa...,unable,to solve Should we add it as an XFAIL for now ?,I have added a check so as to solve the inequa...,NaN,NaN,NaN,NaN
2,issue:openment:22f8203e5a34b1ef8e4258527ea32d4...,issue:openment:22f8203e5a34b1ef8e4258527ea32d4...,issue:openment:22f8203e5a34b1ef8e4258527ea32d4...,It's due to incorrect calculation of due to an,invalid,". On the branch, 's was calculated using a...",It's due to incorrect calculation of due to a...,NaN,NaN,NaN,NaN
3,issue:openment:95a8e6433c701a80a3fd3841f7a3b48...,issue:openment:95a8e6433c701a80a3fd3841f7a3b48...,issue:openment:95a8e6433c701a80a3fd3841f7a3b48...,The only thing is that there are already and ...,individual,methods are better,The only thing is that there are already and ...,NaN,NaN,NaN,NaN
4,issue:openment:0a6e8751019a922060beeda0c854df0...,issue:openment:0a6e8751019a922060beeda0c854df0...,issue:openment:0a6e8751019a922060beeda0c854df0...,", I agree,",incremental,approach will be easier for both of us. I thi...,", I agree, incremental approach will be easier...",NaN,NaN,NaN,NaN
5,issue:openment:d8226a3f93e4b1f0885086ae054a84e...,issue:openment:d8226a3f93e4b1f0885086ae054a84e...,issue:openment:d8226a3f93e4b1f0885086ae054a84e...,my system but I'm getting certain errors.\r ##...,headless,at first and for firefox.\r Used to run pha...,I was trying to run the tests on my system but...,NaN,NaN,NaN,NaN
6,issue:openment:001e9c8d166d061372c32f4eed1c498...,issue:openment:001e9c8d166d061372c32f4eed1c498...,issue:openment:001e9c8d166d061372c32f4eed1c498...,"y the issue, I will investigate the discrepanc...",increase,"in MSE, I think as it updates the weights usi...",You're welcome! I apologize for the errors and...,NaN,NaN,NaN,NaN
7,issue:openment:f832a52e404a9fc2fc1c7fab88991c6...,issue:openment:f832a52e404a9fc2fc1c7fab88991c6...,issue:openment:f832a52e404a9fc2fc1c7fab88991c6...,Look at how has been implemented. Is it possi...,uniformity,?,Look at how has been implemented. Is it possi...,NaN,NaN,NaN,NaN
8,issue:openment:ca00655fc45838a94ff27a09afae1e4...,issue:openment:ca00655fc45838a94ff27a09afae1e4...,issue:openment:ca00655fc45838a94ff27a09afae1e4...,I don't think that the implicit regions like E...,inwards,"or outwards.\r \r Yes, we should have a keywo...",I don't think that the implicit regions like E...,NaN,NaN,NaN,NaN
9,issue:openment:129cd1cc8ee9525adab509edadae8a8...,issue:openment:129cd1cc8ee9525adab509edadae8a8...,issue:openment:129cd1cc8ee9525adab509edadae8a8...,ew lines. I think the problem here is a lot of...,indentation,. Other than that I added tests for all the li...,Thanks a lot for the suggestion. But I think I...,NaN,NaN,NaN,NaN
